# Week 5: Exercise 9 - Provider Factory

**Goal:** Build an abstraction layer for multiple LLM providers.


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    api_key=os.environ.get("GEMINI_API_KEY"),
)
print("API client ready")


API client ready


## Step 1: The Abstract Base Class


In [ ]:
from abc import ABC, abstractmethod

class LLMProvider(ABC):
    """Abstract base for LLM providers."""

    @abstractmethod
    def chat(self, messages, **kwargs):
        """Make a chat request, return raw response."""
        pass

    @abstractmethod
    def parse_response(self, response):
        """Parse response into {content, tool_calls, usage}."""
        pass


## Step 2: Implement GeminiProvider

**TODO:** Implement `__init__`, `chat`, and `parse_response`.


In [ ]:
import os
from openai import OpenAI

class GeminiProvider(LLMProvider):
    """TODO: Implement Gemini provider."""

    def __init__(self, api_key=None, model=None):
        """TODO: Initialize OpenAI client with Gemini base_url."""
        # Hint: base_url="https://generativelanguage.googleapis.com/v1beta/openai"
        #       self.model = model or os.environ.get("GEMINI_MODEL")
        self._client = OpenAI(
            base_url = "https://generativelanguage.googleapis.com/v1beta/openai",
            api_key = api_key or os.environ.get("GEMINI_API_KEY")
        )
        self._model = model or os.environ.get("GEMINI_3.6_MODEL")

    def chat(self, messages, model=None, tools=None, **kwargs):
        """TODO: Call chat completions."""
        # Hint: self.client.chat.completions.create(model=..., messages=..., tools=..., **kwargs)
        #       NO try/except here — let errors propagate to the FallbackChain
        response = self._client.chat.completions.create(
            model = model or self._model,
            messages = messages,
            tools = tools,
            **kwargs
        )
        return response

    def parse_response(self, response):
        """TODO: Convert to standard format."""
        # Hint: msg = response.choices[0].message
        #       return {"content": ..., "tool_calls": ..., "usage": {...}}
        msg = response.choices[0].message
        return {
            "content": msg.content,
            "tool_calls": msg.tool_calls,
            "usage": {
                "prompt_tokens": getattr(response.usage, "prompt_tokens", 0),
                "completion_tokens": getattr(response.usage, "completion_tokens", 0)
            }
        }


## Step 3: Implement ProviderFactory.create


In [4]:
class ProviderFactory:
    _providers = {}

    @classmethod
    def register(cls, name, provider_class):
        cls._providers[name] = provider_class

    @classmethod
    def create(cls, name, **kwargs):
        """TODO: Create and return provider instance."""
        # Hint: look up cls._providers.get(name)
        #       raise ValueError if not found
        #       return provider_class(**kwargs)
        if name in cls._providers:
            return cls._providers[name](**kwargs)
        else: raise ValueError(f"Function {name} not found!")

    @classmethod
    def list_providers(cls):
        return list(cls._providers.keys())


## Step 4: Register and Test


In [5]:
# Test the ProviderFactory
ProviderFactory.register("gemini", GeminiProvider)
provider = ProviderFactory.create("gemini")
print("Available providers:", ProviderFactory.list_providers())

messages = [{"role": "user", "content": "Say hello in one word."}]
raw = provider.chat(messages)
parsed = provider.parse_response(raw)
print("Response:", parsed["content"])


Available providers: ['gemini']
Response: Hello


---
# Part 2: Fallback Provider Chain

**Goal:** Automatically switch providers when one fails.

Uses the `GeminiProvider` built in Part 1 above.


## Step 1: ProviderEntry Dataclass


In [6]:
import time
from typing import List
from dataclasses import dataclass

@dataclass
class ProviderEntry:
    provider: object
    model: str
    priority: int = 0
    error_count: int = 0
    last_error_time: float = 0


## Step 2: Implement FallbackChain


In [26]:
class FallbackChain:
    """Try providers in order until one succeeds."""

    def __init__(self):
        self._providers: List[ProviderEntry] = []

    def add_provider(self, provider, model, priority=0):
        """TODO: Add with priority ordering."""
        # Hint: create a ProviderEntry, append, sort by priority
        provider_entry = ProviderEntry(provider, model, priority)
        self._providers.append(provider_entry)
        self._providers.sort(key = lambda e: e.priority)

    def get_healthy_providers(self):
        """TODO: Return providers with fewer than 3 recent errors."""
        # Hint: keep entries where error_count < 3,
        #       OR last error was more than 60 seconds ago
        return [provider_entry for provider_entry in self._providers
                if provider_entry.error_count < 3 or (time.time()-provider_entry.last_error_time > 60)]

    def chat(self, messages, max_retries=2):
        """TODO: Try each provider until one succeeds."""
        # Hint: loop over healthy providers, retry each up to max_retries,
        #       on success return parse_response, on failure record the error
        #       (error_count += 1, last_error_time = time.time())
        #       after ALL loops: return {"error": "All providers failed"}
        healthy_providers = self.get_healthy_providers()
        for provider_entry in healthy_providers:
            for attempt in range(max_retries + 1):
                try:
                    response = provider_entry.provider.chat(messages)

                    return provider_entry.provider.parse_response(response)
                except Exception as E:
                    
                    provider_entry.error_count += 1
                    provider_entry.last_error_time = time.time()
                    retryable = self._is_retryable_error(str(E)) 

                    if not retryable or attempt == max_retries:
                        print(f"Giving up on error {E}")
                        break
                    
                    time.sleep(2 ** attempt)               
                    
        return {"error": "All providers failed"}

    def _is_retryable_error(self, error_str):
        retryable = ["rate limit", "timeout", "502", "503", "504", "429"]
        return any(p in error_str.lower() for p in retryable)


## Test Your Solution

The tests below use the models defined in `../.env`:

- `GEMINI_MODEL` (gemini-flash-latest) — healthy, succeeds
- `GEMINI_2.0_MODEL` (gemini-2.0-flash) — healthy, succeeds
- `GEMINI_BROKEN_MODEL` (gemini-fake-broken-model) — does not exist on the server, guaranteed to fail with 404

**Test 1 (success + fallback):** broken model at priority 1 fails with 404 (non-retryable → skip to next provider immediately), the chain falls through to a healthy model and answers.

**Test 2 (all providers fail):** every provider points at the broken model. The chain tries each, exhausts the chain, and returns `{"error": "All providers failed"}` instead of crashing.


In [27]:
# Test 1: fallback — broken model fails 404, chain falls through to healthy model
chain = FallbackChain()

broken_model = os.environ.get("GEMINI_BROKEN_MODEL")
good_model = os.environ.get("GEMINI_3.6_MODEL")

p1 = GeminiProvider(model=broken_model)
p2 = GeminiProvider(model=good_model)
chain.add_provider(p1, broken_model, priority=1)   # tried first, will 404
chain.add_provider(p2, good_model, priority=2)     # picks up the slack

assert len(chain._providers) == 2
print("Providers in chain:", len(chain._providers))
print("Healthy providers:", len(chain.get_healthy_providers()))

messages = [{"role": "user", "content": "Say hello in one word."}]
result = chain.chat(messages)
print("Result:", result.get("content", result))

assert "error" not in result, f"chain should have fallen back, got: {result}"
print("Test 1 passed: fallback to healthy provider works.")


Providers in chain: 2
Healthy providers: 2
Giving up on error Error code: 404 - [{'error': {'code': 404, 'message': 'models/gemini-fake-broken-model is not found for API version v1main, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}]
Result: Hello.
Test 1 passed: fallback to healthy provider works.


In [28]:
# Test 2: all providers fail — chain returns error dict instead of crashing
chain2 = FallbackChain()

p3 = GeminiProvider(model=broken_model)
p4 = GeminiProvider(model=broken_model)
chain2.add_provider(p3, broken_model, priority=1)
chain2.add_provider(p4, broken_model, priority=2)

messages = [{"role": "user", "content": "Say hello in one word."}]
result2 = chain2.chat(messages)
print("Result:", result2)

assert result2.get("error") == "All providers failed", f"expected error dict, got: {result2}"
assert chain2._providers[0].error_count > 0, "failure should have been recorded on the entry"
print("Test 2 passed: all-fail returns error, error_count tracked.")


Giving up on error Error code: 404 - [{'error': {'code': 404, 'message': 'models/gemini-fake-broken-model is not found for API version v1main, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}]
Giving up on error Error code: 404 - [{'error': {'code': 404, 'message': 'models/gemini-fake-broken-model is not found for API version v1main, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}]
Result: {'error': 'All providers failed'}
Test 2 passed: all-fail returns error, error_count tracked.


## Key Takeaways
- The ABC defines the contract; concrete classes implement it
- `parse_response` normalizes outputs into one standard format
- Fallback chains make agents resilient
- Priority ordering controls which provider is tried first
